In [2]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules

In [3]:
# Dummy transaction dataset
data = {'Milk': [1,0,1,1,0,1], 'Bread': [1,1,1,1,1,1], 'Butter': [0,1,1,0,1,1], 'Jam': [0,1,0,0,1,1]}
df = pd.DataFrame(data)

In [4]:
df

,Milk,Bread,Butter,Jam
0,1,1,0,0
1,0,1,1,1
2,1,1,1,0
3,1,1,0,0
4,0,1,1,1
5,1,1,1,1


In [9]:
# Hyperparameter: min_support dictates the threshold for frequent itemsets
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)

/Users/thejus/uv-envs/envy-tf/.venv/lib/python3.11/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [10]:
frequent_itemsets

,support,itemsets
0,0.666667,frozenset({Milk})
1,1.000000,frozenset({Bread})
2,0.666667,frozenset({Butter})
3,0.500000,frozenset({Jam})
4,0.666667,"frozenset({Bread, Milk})"
5,0.333333,"frozenset({Butter, Milk})"
6,0.666667,"frozenset({Bread, Butter})"
7,0.500000,"frozenset({Bread, Jam})"
8,0.500000,"frozenset({Jam, Butter})"
9,0.333333,"frozenset({Bread, Butter, Milk})"


In [11]:
# Hyperparameter: min_threshold (confidence) dictates rule strictness
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

/Users/thejus/uv-envs/envy-tf/.venv/lib/python3.11/site-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [12]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({Bread}),frozenset({Milk}),1.000000,0.666667,0.666667,0.666667,1.0,1.0,0.000000,1.0,0.000000,0.666667,0.0,0.833333
1,frozenset({Milk}),frozenset({Bread}),0.666667,1.000000,0.666667,1.000000,1.0,1.0,0.000000,inf,0.000000,0.666667,0.0,0.833333
2,frozenset({Bread}),frozenset({Butter}),1.000000,0.666667,0.666667,0.666667,1.0,1.0,0.000000,1.0,0.000000,0.666667,0.0,0.833333
3,frozenset({Butter}),frozenset({Bread}),0.666667,1.000000,0.666667,1.000000,1.0,1.0,0.000000,inf,0.000000,0.666667,0.0,0.833333
4,frozenset({Jam}),frozenset({Bread}),0.500000,1.000000,0.500000,1.000000,1.0,1.0,0.000000,inf,0.000000,0.500000,0.0,0.750000
5,frozenset({Jam}),frozenset({Butter}),0.500000,0.666667,0.500000,1.000000,1.5,1.0,0.166667,inf,0.666667,0.750000,1.0,0.875000
6,frozenset({Butter}),frozenset({Jam}),0.666667,0.500000,0.500000,0.750000,1.5,1.0,0.166667,2.0,1.000000,0.750000,0.5,0.875000
7,"frozenset({Butter, Milk})",frozenset({Bread}),0.333333,1.000000,0.333333,1.000000,1.0,1.0,0.000000,inf,0.000000,0.333333,0.0,0.666667
8,"frozenset({Bread, Butter})",frozenset({Jam}),0.666667,0.500000,0.500000,0.750000,1.5,1.0,0.166667,2.0,1.000000,0.750000,0.5,0.875000
9,"frozenset({Bread, Jam})",frozenset({Butter}),0.500000,0.666667,0.500000,1.000000,1.5,1.0,0.166667,inf,0.666667,0.750000,1.0,0.875000


In [13]:
# Visualization: Directed Graph of Rules
plt.figure(figsize=(8, 6))
G = nx.DiGraph()

for idx, row in rules.iterrows():
    antecedent = list(row['antecedents'])[0]
    consequent = list(row['consequents'])[0]
    weight = row['lift']
    G.add_edge(antecedent, consequent, weight=weight)

# Draw the network
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=12, font_weight='bold', edge_color='gray')
edge_labels = nx.get_edge_attributes(G, 'weight')
# Format lift values to 2 decimal places for cleaner display
edge_labels = {k: f"Lift: {v:.2f}" for k, v in edge_labels.items()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

plt.title("Association Rules Network (Edge thickness/labels represent Lift)")
plt.show()

`min_support`: The minimum frequency required for an itemset to be considered "popular." (Too low = computational explosion; too high = no rules).

`min_confidence`: How often the rule has been found to be true. (The conditional probability).